In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/09 17:51:16 WARN Utils: Your hostname, codespaces-37f61e, resolves to a loopback address: 127.0.0.1; using 10.0.3.221 instead (on interface eth0)
26/05/09 17:51:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 17:51:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

26/05/09 17:51:23 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

```
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
```

<h3>Resilient Distributed Dataset</h3>

<p>Think of an RDD as a giant, distributed list of objects that is</p>
<p>split up across many different computers in a cluster so they can be processed in parallel</p>

pretty much a specialized dataframe you get with spark..

<h3>Why use RDDs today? (RDDs vs. DataFrames)</h3>

<p>While DataFrames (which look like SQL tables) are the modern standard </p>
<p>because they are highly optimized by Spark's "Catalyst Optimizer," </p>

<p>RDDs are still useful in specific scenarios:</p>

<p>Low-level control: When you need to control exactly how data is partitioned or distributed.</p>

<p>Unstructured data: If you are dealing with raw streams of text or media where a schema (rows and columns) doesn't make sense.</p>

Functional Programming: If you prefer working with Scala/Python objects and lambda functions rather than SQL-like expressions.

In [3]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [4]:
from datetime import datetime

In [5]:
start = datetime(year=2021, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

using function with rdds

In [6]:
rows = rdd.take(10)
row = rows[0]

In [7]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 23, 13, 10, 15), PULocationID=74, total_amount=44.97)

doing transformttions by row on rdds

<p><strong>prepare_for_grouping(row)</strong> - transforms each row from the rdd int a key-value pair for grouping</p>
<p>This groups rows by the same hour and zone</p>

In [ ]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [9]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [10]:
rdd \
.filter(filter_outliers) \
.map(prepare_for_grouping) \
.reduceByKey(calculate_revenue) \
.take(10)

[((datetime.datetime(2021, 6, 24, 22, 0), 41), (122.05, 8)),
 ((datetime.datetime(2021, 6, 27, 20, 0), 75), (118.32, 7)),
 ((datetime.datetime(2021, 6, 23, 19, 0), 22), (63.3, 1)),
 ((datetime.datetime(2021, 6, 29, 12, 0), 24), (23.97, 2)),
 ((datetime.datetime(2021, 6, 18, 17, 0), 52), (90.79, 1)),
 ((datetime.datetime(2021, 6, 11, 14, 0), 167), (37.1, 2)),
 ((datetime.datetime(2021, 6, 7, 16, 0), 213), (44.3, 2)),
 ((datetime.datetime(2021, 6, 14, 21, 0), 93), (140.39999999999998, 3)),
 ((datetime.datetime(2021, 6, 18, 11, 0), 213), (67.66, 3)),
 ((datetime.datetime(2021, 6, 28, 9, 0), 197), (54.8, 1))]

In [11]:
from collections import namedtuple

In [12]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [13]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [14]:
rdd \
.filter(filter_outliers) \
.map(prepare_for_grouping) \
.reduceByKey(calculate_revenue) \
.map(unwrap) \
.take(10)

[RevenueRow(hour=datetime.datetime(2021, 6, 24, 22, 0), zone=41, revenue=122.05, count=8),
 RevenueRow(hour=datetime.datetime(2021, 6, 27, 20, 0), zone=75, revenue=118.32, count=7),
 RevenueRow(hour=datetime.datetime(2021, 6, 23, 19, 0), zone=22, revenue=63.3, count=1),
 RevenueRow(hour=datetime.datetime(2021, 6, 29, 12, 0), zone=24, revenue=23.97, count=2),
 RevenueRow(hour=datetime.datetime(2021, 6, 18, 17, 0), zone=52, revenue=90.79, count=1),
 RevenueRow(hour=datetime.datetime(2021, 6, 11, 14, 0), zone=167, revenue=37.1, count=2),
 RevenueRow(hour=datetime.datetime(2021, 6, 7, 16, 0), zone=213, revenue=44.3, count=2),
 RevenueRow(hour=datetime.datetime(2021, 6, 14, 21, 0), zone=93, revenue=140.39999999999998, count=3),
 RevenueRow(hour=datetime.datetime(2021, 6, 18, 11, 0), zone=213, revenue=67.66, count=3),
 RevenueRow(hour=datetime.datetime(2021, 6, 28, 9, 0), zone=197, revenue=54.8, count=1)]

In [15]:
from pyspark.sql import types

In [16]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [17]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema) 

In [18]:
df_result.write.parquet('tmp/green-revenue')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/workspaces/DataEngineering-DataTalksClub/06-batch/code/tmp/green-revenue already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [ ]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [ ]:
import pandas as pd

In [ ]:
rows = duration_rdd.take(10)

In [ ]:
df = pd.DataFrame(rows, columns=columns)

In [ ]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [ ]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [ ]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [ ]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [ ]:
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
|63.849999999999994|
|              40.0|
|              6.35|
|              6.25|
| 9.200000000000001|
|               3.8|
|16.599999999999998|
|             11.05|
|               4.5|
|              30.5|
|               8.7|
|5.8999999999999995|
|              11.0|
|              15.2|
|              4.25|
|25.299999999999997|
|7.8500000000000005|
|              34.0|
| 5.300000000000001|
|              6.15|
+------------------+
only showing top 20 rows


Traceback (most recent call last):                                              
  File "/workspaces/DataEngineering-DataTalksClub/venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/workspaces/DataEngineering-DataTalksClub/venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
